In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 2 — BASELINE SIMILARITY MATRIX
# Inputs:  outputs/02_returns_matrix.parquet
#          outputs/subperiod_labels.csv
#          outputs/phase1_fat_tail_params.csv
#          outputs/03_coin_metadata.csv
# Outputs: outputs/04_correlation_matrix_full.parquet   ← Pearson + Spearman
#          outputs/05_correlation_matrix_btcadj.parquet ← BTC-beta adjusted
#          outputs/phase2_rolling_corr_30d.parquet      ← rolling 30d
#          outputs/phase2_rolling_corr_90d.parquet      ← rolling 90d
#          outputs/phase2_subperiod_corr.parquet        ← corr per subperiod
#          outputs/phase2_clustering_baseline.csv       ← first-pass clusters
#          outputs/phase2_btc_betas.csv                 ← beta + residuals per coin
# ═══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

# Stats
from scipy.stats import spearmanr, pearsonr
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.linear_model import HuberRegressor   # robust BTC-beta regression

# Multiple testing (BH-FDR from Phase 0.5)
from statsmodels.stats.multitest import multipletests

# Visualisation
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — safe for all environments
import matplotlib.pyplot as plt
import seaborn as sns

Path("outputs").mkdir(exist_ok=True)
Path("outputs/figures").mkdir(exist_ok=True)


# ═══════════════════════════════════════════════════════════════════════════════
# LOAD INPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("Loading Phase 1 outputs...")

log_returns  = pd.read_parquet("outputs/02_returns_matrix.parquet")
fat_tail_df  = pd.read_csv("outputs/phase1_fat_tail_params.csv")
coin_meta    = pd.read_csv("outputs/03_coin_metadata.csv")
subperiod_df = pd.read_csv("outputs/subperiod_labels.csv", parse_dates=["date"])

coins = log_returns.columns.tolist()
N     = len(coins)
print(f"  Coins            : {N}")
print(f"  Return matrix    : {log_returns.shape}")
print(f"  Date range       : {log_returns.index[0]}  →  {log_returns.index[-1]}")


# ═══════════════════════════════════════════════════════════════════════════════
# SUBPERIOD CONSOLIDATION
# Phase 1 produced 22 monthly subperiods — too granular for reliable statistics.
# We consolidate to 4 meaningful regimes based on the dominant consensus breaks:
#
#   Period_A : Mar 2024 → Oct 2024   (pre-election, post-halving consolidation)
#   Period_B : Nov 2024 → Jan 2025   (post-election rally — strongest break)
#   Period_C : Feb 2025 → Sep 2025   (2025 mid-cycle)
#   Period_D : Oct 2025 → Mar 2026   (late cycle + correction)
#
# Rationale: Nov 2024 (78% of coins), Jan 2025 (63%), Oct 2025 (63%) are the
# three dominant consensus breaks. Everything between them is one coherent regime.
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Consolidating subperiods ────────────────────────────────────────────────")

SUBPERIOD_BREAKS = [
    pd.Timestamp("2024-11-01"),   # post-election rally start
    pd.Timestamp("2025-02-01"),   # rally peak / consolidation
    pd.Timestamp("2025-10-01"),   # late-cycle shift
]

def assign_consolidated_period(date):
    if date < SUBPERIOD_BREAKS[0]:
        return "Period_A"   # Mar 2024 – Oct 2024
    elif date < SUBPERIOD_BREAKS[1]:
        return "Period_B"   # Nov 2024 – Jan 2025
    elif date < SUBPERIOD_BREAKS[2]:
        return "Period_C"   # Feb 2025 – Sep 2025
    else:
        return "Period_D"   # Oct 2025 – Mar 2026

subperiod_df["consolidated"] = pd.to_datetime(
    subperiod_df["date"]
).apply(assign_consolidated_period)

# Print sizes
for p in ["Period_A", "Period_B", "Period_C", "Period_D"]:
    mask  = subperiod_df["consolidated"] == p
    dates = subperiod_df.loc[mask, "date"]
    print(f"  {p} : {mask.sum():3d} days  ({dates.iloc[0].date()} → {dates.iloc[-1].date()})")

# Save consolidated labels
subperiod_df.to_csv("outputs/subperiod_labels.csv", index=False)


# ═══════════════════════════════════════════════════════════════════════════════
# HELPER — BH-FDR CORRECTION  (imported pattern from Phase 0.5)
# ═══════════════════════════════════════════════════════════════════════════════
def apply_bh_correction(pvalues: np.ndarray, q: float = 0.05) -> dict:
    reject, pvals_adj, _, _ = multipletests(pvalues, alpha=q, method="fdr_bh")
    return {"adjusted_pvalues": pvals_adj, "reject": reject,
            "n_significant": reject.sum()}


# ═══════════════════════════════════════════════════════════════════════════════
# 2.1  STATIC CORRELATION — PEARSON + SPEARMAN (full period)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 2.1 │ Static Correlation Matrices ─────────────────────────────────")

# Why both Pearson and Spearman?
# Pearson measures LINEAR correlation and is sensitive to outliers and fat tails.
# Spearman is rank-based — it sees a monotone relationship even if non-linear,
# and is robust to the extreme daily moves we confirmed in Phase 1.2.
# For a universe where median ν = 4.11, Spearman is the more trustworthy measure.
# We compute both: Pearson as the industry-standard benchmark, Spearman as our
# primary working measure.

def compute_correlation_matrices(returns_df: pd.DataFrame) -> tuple:
    """
    Computes Pearson and Spearman correlation matrices.
    Uses pairwise complete observations — handles NaNs from late-starting coins.
    Also returns p-value matrices with BH correction applied.
    """
    coin_list = returns_df.columns.tolist()
    n         = len(coin_list)

    pearson_mat  = np.full((n, n), np.nan)
    spearman_mat = np.full((n, n), np.nan)
    pearson_pval = np.full((n, n), np.nan)
    spearman_pval= np.full((n, n), np.nan)

    np.fill_diagonal(pearson_mat,  1.0)
    np.fill_diagonal(spearman_mat, 1.0)
    np.fill_diagonal(pearson_pval,  0.0)
    np.fill_diagonal(spearman_pval, 0.0)

    raw_p_pearson  = []
    raw_p_spearman = []
    pair_indices   = []

    for i, j in combinations(range(n), 2):
        xi = returns_df.iloc[:, i]
        xj = returns_df.iloc[:, j]
        mask = xi.notna() & xj.notna()
        xi_c, xj_c = xi[mask].values, xj[mask].values

        if len(xi_c) < 30:
            continue

        pr, pp = pearsonr(xi_c, xj_c)
        sr, sp = spearmanr(xi_c, xj_c)

        pearson_mat[i, j]  = pearson_mat[j, i]  = pr
        spearman_mat[i, j] = spearman_mat[j, i] = sr

        raw_p_pearson.append(pp)
        raw_p_spearman.append(sp)
        pair_indices.append((i, j))

    # Apply BH correction
    bh_p  = apply_bh_correction(np.array(raw_p_pearson))
    bh_s  = apply_bh_correction(np.array(raw_p_spearman))

    for k, (i, j) in enumerate(pair_indices):
        pearson_pval[i, j]  = pearson_pval[j, i]  = bh_p["adjusted_pvalues"][k]
        spearman_pval[i, j] = spearman_pval[j, i] = bh_s["adjusted_pvalues"][k]

    pearson_df  = pd.DataFrame(pearson_mat,  index=coin_list, columns=coin_list)
    spearman_df = pd.DataFrame(spearman_mat, index=coin_list, columns=coin_list)
    pp_df       = pd.DataFrame(pearson_pval, index=coin_list, columns=coin_list)
    sp_df       = pd.DataFrame(spearman_pval,index=coin_list, columns=coin_list)

    return pearson_df, spearman_df, pp_df, sp_df, bh_p["n_significant"], bh_s["n_significant"]


print("  Computing full-period Pearson + Spearman...")
pearson_full, spearman_full, pp_full, sp_full, n_sig_p, n_sig_s = \
    compute_correlation_matrices(log_returns)

# ─── Sanity check — BTC–ETH correlation should be > 0.7 ─────────────────────
btc_eth_pearson  = pearson_full.loc["bitcoin", "ethereum"]
btc_eth_spearman = spearman_full.loc["bitcoin", "ethereum"]
print(f"\n  Sanity check — BTC–ETH correlation:")
print(f"    Pearson  : {btc_eth_pearson:.4f}  (expected > 0.70)")
print(f"    Spearman : {btc_eth_spearman:.4f}  (expected > 0.70)")

if btc_eth_pearson < 0.70:
    print("  ⚠️  BTC–ETH Pearson below 0.70 — inspect return data")
else:
    print("  ✅ Sanity check passed")

# ─── Distribution summary ────────────────────────────────────────────────────
upper_tri = spearman_full.values[np.triu_indices(N, k=1)]
upper_tri = upper_tri[~np.isnan(upper_tri)]

print(f"\n  Spearman correlation distribution (all {len(upper_tri):,} pairs):")
print(f"    Mean   : {np.mean(upper_tri):.4f}")
print(f"    Median : {np.median(upper_tri):.4f}")
print(f"    Std    : {np.std(upper_tri):.4f}")
print(f"    Min    : {np.min(upper_tri):.4f}")
print(f"    Max    : {np.max(upper_tri):.4f}")
print(f"    Pairs ρ > 0.70  : {(upper_tri > 0.70).sum():,}")
print(f"    Pairs ρ > 0.50  : {(upper_tri > 0.50).sum():,}")
print(f"    Pairs ρ < 0.20  : {(upper_tri < 0.20).sum():,}")
print(f"\n  BH-significant pairs — Pearson  : {n_sig_p:,} / {len(upper_tri):,}")
print(f"  BH-significant pairs — Spearman : {n_sig_s:,} / {len(upper_tri):,}")

# ─── Heatmap ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(
    spearman_full.fillna(0),
    cmap="RdYlGn", center=0, vmin=-1, vmax=1,
    xticklabels=False, yticklabels=False,
    ax=ax, cbar_kws={"label": "Spearman ρ"}
)
ax.set_title("Spearman Correlation Matrix — Full Period (125 coins)", fontsize=14)
plt.tight_layout()
plt.savefig("outputs/figures/phase2_spearman_heatmap.png", dpi=150)
plt.close()
print("\n  ✅ Saved: phase2_spearman_heatmap.png")


# ═══════════════════════════════════════════════════════════════════════════════
# 2.2  ROLLING CORRELATION — 30d and 90d
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 2.2 │ Rolling Correlation ─────────────────────────────────────────")

# We compute rolling correlation for the top 20 most liquid pairs by median
# daily volume. Computing all 7,750 pairs × 729 days is memory-intensive and
# the distribution plot is what matters here — not individual pair tracking.
# For Phase 7 composite score we use the 90d rolling value at the end of the
# window (most recent 90-day correlation) for all pairs.

def rolling_corr_endpoint(returns_df: pd.DataFrame,
                           window: int) -> pd.DataFrame:
    """
    For each pair, computes the Pearson correlation over the most recent
    `window` trading days. Returns a (N × N) matrix of endpoint values.
    This is the Phase 7 input — "what is the current rolling correlation?"
    """
    coin_list = returns_df.columns.tolist()
    n         = len(coin_list)
    mat       = np.full((n, n), np.nan)
    np.fill_diagonal(mat, 1.0)

    recent = returns_df.iloc[-window:]  # most recent window

    for i, j in combinations(range(n), 2):
        xi = recent.iloc[:, i]
        xj = recent.iloc[:, j]
        mask = xi.notna() & xj.notna()
        if mask.sum() < window * 0.7:   # need at least 70% of window filled
            continue
        r, _ = pearsonr(xi[mask].values, xj[mask].values)
        mat[i, j] = mat[j, i] = r

    return pd.DataFrame(mat, index=coin_list, columns=coin_list)


def rolling_corr_timeseries(returns_df: pd.DataFrame,
                             coin_a: str, coin_b: str,
                             window: int) -> pd.Series:
    """Rolling Pearson correlation between two coins over time."""
    return returns_df[coin_a].rolling(window).corr(returns_df[coin_b])


print(f"  Computing 30d endpoint correlation matrix...")
rolling_30d = rolling_corr_endpoint(log_returns, window=30)

print(f"  Computing 90d endpoint correlation matrix...")
rolling_90d = rolling_corr_endpoint(log_returns, window=90)

# Distribution of 90d rolling correlations
upper_90d = rolling_90d.values[np.triu_indices(N, k=1)]
upper_90d = upper_90d[~np.isnan(upper_90d)]

print(f"\n  90d rolling correlation distribution ({len(upper_90d):,} pairs):")
print(f"    Mean   : {np.mean(upper_90d):.4f}")
print(f"    Median : {np.median(upper_90d):.4f}")
print(f"    Pairs ρ > 0.70 : {(upper_90d > 0.70).sum():,}")
print(f"    Pairs ρ < 0.20 : {(upper_90d < 0.20).sum():,}")

# Time-series plot for key pairs
key_pairs = [
    ("bitcoin",  "ethereum"),
    ("bitcoin",  "solana"),
    ("ethereum", "solana"),
    ("bitcoin",  "pax-gold"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (ca, cb) in enumerate(key_pairs):
    if ca not in log_returns.columns or cb not in log_returns.columns:
        continue
    ts_30 = rolling_corr_timeseries(log_returns, ca, cb, 30)
    ts_90 = rolling_corr_timeseries(log_returns, ca, cb, 90)
    ax = axes[idx]
    ax.plot(ts_30.index, ts_30.values, alpha=0.5, label="30d", color="steelblue")
    ax.plot(ts_90.index, ts_90.values, alpha=0.9, label="90d", color="darkblue")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.axhline(0.7, color="green", linestyle=":", linewidth=0.8, label="ρ=0.70")

    # Mark consolidated subperiod breaks
    for brk in SUBPERIOD_BREAKS:
        ax.axvline(brk, color="red", linestyle="--", alpha=0.5, linewidth=0.8)

    ax.set_title(f"{ca} vs {cb}", fontsize=11)
    ax.set_ylim(-1, 1)
    ax.legend(fontsize=8)
    ax.set_xlabel("")

plt.suptitle("Rolling Correlation — Key Pairs (red dashed = regime breaks)", fontsize=13)
plt.tight_layout()
plt.savefig("outputs/figures/phase2_rolling_corr_keypairs.png", dpi=150)
plt.close()
print("\n  ✅ Saved: phase2_rolling_corr_keypairs.png")


# ═══════════════════════════════════════════════════════════════════════════════
# 2.3  BTC-BETA ADJUSTED CORRELATION
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 2.3 │ BTC-Beta Adjusted Correlation ───────────────────────────────")

# Why robust regression?
# BTC has ν = 3.71 — extreme fat tails confirmed in Phase 1.2.
# OLS minimises squared errors, making it highly sensitive to the very large
# outlier days that are common in crypto. A single -20% BTC day can dominate
# the beta estimate. We use Huber regression which down-weights outliers
# automatically, giving us a more stable beta estimate.

# Step 1: For each coin, regress returns on BTC returns → extract beta + residuals
# Step 2: Residuals represent the coin's "BTC-independent" movement
# Step 3: Correlate residuals → partial correlation after removing BTC factor
# This separates "they move together BECAUSE of BTC" from "they move
# together independently of BTC" — both matter but must be distinguished.

if "bitcoin" not in log_returns.columns:
    raise ValueError("Bitcoin not in return matrix — required for BTC-beta adjustment")

btc_returns = log_returns["bitcoin"].values

beta_rows   = []
residuals   = pd.DataFrame(index=log_returns.index, columns=coins, dtype=float)

print("  Fitting Huber regression (robust BTC-beta) for all coins...")

for coin in coins:
    if coin == "bitcoin":
        residuals["bitcoin"] = log_returns["bitcoin"]  # BTC residual = itself
        beta_rows.append({"coin_id": coin, "btc_beta": 1.0, "intercept": 0.0,
                          "n_obs": log_returns["bitcoin"].notna().sum()})
        continue

    coin_ret = log_returns[coin]
    mask     = coin_ret.notna() & pd.Series(btc_returns,
                                             index=log_returns.index).notna()
    xi = btc_returns[mask.values].reshape(-1, 1)
    yi = coin_ret[mask].values

    if len(yi) < 60:
        residuals[coin] = np.nan
        beta_rows.append({"coin_id": coin, "btc_beta": np.nan,
                          "intercept": np.nan, "n_obs": len(yi)})
        continue

    try:
        huber = HuberRegressor(epsilon=1.35, max_iter=300)
        huber.fit(xi, yi)
        beta    = huber.coef_[0]
        alpha   = huber.intercept_
        yi_pred = huber.predict(xi)
        resid   = yi - yi_pred

        resid_series                  = pd.Series(np.nan, index=log_returns.index)
        resid_series[mask]            = resid
        residuals[coin]               = resid_series

        beta_rows.append({"coin_id": coin, "btc_beta": round(beta, 4),
                          "intercept": round(alpha, 6), "n_obs": len(yi)})
    except Exception:
        residuals[coin] = np.nan
        beta_rows.append({"coin_id": coin, "btc_beta": np.nan,
                          "intercept": np.nan, "n_obs": len(yi)})

beta_df = pd.DataFrame(beta_rows)

# ─── Beta distribution summary ───────────────────────────────────────────────
betas = beta_df["btc_beta"].dropna()
print(f"\n  BTC-beta distribution across universe:")
print(f"    Mean   : {betas.mean():.3f}")
print(f"    Median : {betas.median():.3f}")
print(f"    Min    : {betas.min():.3f}  ({beta_df.loc[betas.idxmin(), 'coin_id']})")
print(f"    Max    : {betas.max():.3f}  ({beta_df.loc[betas.idxmax(), 'coin_id']})")
print(f"    Coins β > 1.5 (high sensitivity) : {(betas > 1.5).sum()}")
print(f"    Coins β < 0.3 (low sensitivity)  : {(betas < 0.3).sum()}")

# ─── Compute correlation on residuals ────────────────────────────────────────
print("\n  Computing BTC-adjusted Spearman correlation on residuals...")
pearson_adj, spearman_adj, _, _, _, n_sig_adj = \
    compute_correlation_matrices(residuals)

# ─── Compare: raw vs BTC-adjusted ────────────────────────────────────────────
upper_raw = spearman_full.values[np.triu_indices(N, k=1)]
upper_adj = spearman_adj.values[np.triu_indices(N, k=1)]
mask_both = ~np.isnan(upper_raw) & ~np.isnan(upper_adj)

print(f"\n  Correlation drop after BTC-adjustment:")
print(f"    Raw Spearman median      : {np.median(upper_raw[mask_both]):.4f}")
print(f"    Adjusted Spearman median : {np.median(upper_adj[mask_both]):.4f}")
print(f"    Median drop              : {np.median(upper_raw[mask_both] - upper_adj[mask_both]):.4f}")
print(f"    Pairs still ρ_adj > 0.50 : {(upper_adj[mask_both] > 0.50).sum():,}")


# ═══════════════════════════════════════════════════════════════════════════════
# 2.4  SUBPERIOD CORRELATIONS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 2.4 │ Subperiod Correlations ──────────────────────────────────────")

# Compute Spearman correlation within each consolidated subperiod.
# This is not the regime-conditional analysis (Phase 4) — that uses
# Markov-identified bull/bear regimes. This uses our structural break
# subperiods to check whether similarity is stable across market regimes.

subperiod_corr = {}
period_names   = ["Period_A", "Period_B", "Period_C", "Period_D"]

for period in period_names:
    period_dates  = subperiod_df.loc[
        subperiod_df["consolidated"] == period, "date"
    ].values
    period_dates  = pd.to_datetime(period_dates)
    period_returns= log_returns.loc[
        log_returns.index.isin(period_dates)
    ]

    if len(period_returns) < 20:
        print(f"  ⚠️  {period} has only {len(period_returns)} days — skipping")
        continue

    _, sp_period, _, _, _, _ = compute_correlation_matrices(period_returns)
    subperiod_corr[period]   = sp_period

    upper = sp_period.values[np.triu_indices(N, k=1)]
    upper = upper[~np.isnan(upper)]
    print(f"  {period}: {len(period_returns):3d} days │ "
          f"median ρ = {np.median(upper):.3f} │ "
          f"ρ>0.70: {(upper > 0.70).sum():,} pairs")


# ─── Stability check: correlation change across subperiods ───────────────────
print("\n  Checking correlation stability across subperiods...")

# For each pair, measure the range of Spearman ρ across the 4 subperiods.
# High range = similarity is regime-dependent (unstable).
# Low range  = similarity is regime-stable (more trustworthy).

stability_rows = []
for i, j in combinations(range(N), 2):
    ci, cj = coins[i], coins[j]
    vals = []
    for period in period_names:
        if period not in subperiod_corr:
            continue
        v = subperiod_corr[period].iloc[i, j]
        if not np.isnan(v):
            vals.append(v)
    if len(vals) >= 3:
        stability_rows.append({
            "coin_a"        : ci,
            "coin_b"        : cj,
            "corr_mean"     : round(np.mean(vals), 4),
            "corr_min"      : round(np.min(vals), 4),
            "corr_max"      : round(np.max(vals), 4),
            "corr_range"    : round(np.max(vals) - np.min(vals), 4),
            "n_periods"     : len(vals),
        })

stability_df = pd.DataFrame(stability_rows)

if len(stability_df):
    print(f"  Pairs with data across ≥3 subperiods: {len(stability_df):,}")
    print(f"  Median correlation range across periods : {stability_df['corr_range'].median():.3f}")
    print(f"  Pairs with range > 0.40 (unstable)     : {(stability_df['corr_range'] > 0.40).sum():,}")
    print(f"  Pairs with range < 0.15 (stable)       : {(stability_df['corr_range'] < 0.15).sum():,}")


# ═══════════════════════════════════════════════════════════════════════════════
# 2.5  FIRST-PASS HIERARCHICAL CLUSTERING
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 2.5 │ First-Pass Hierarchical Clustering ─────────────────────────")

# Distance matrix = 1 - |ρ_spearman|
# We use absolute value so that strongly negatively correlated coins
# are also considered close (they move in lockstep, just inversely).
# Ward linkage minimises within-cluster variance.

# Handle NaNs in distance matrix — replace with max distance (1.0)
spearman_vals = spearman_full.values.copy()
spearman_vals = np.where(np.isnan(spearman_vals), 0.0, spearman_vals)
np.fill_diagonal(spearman_vals, 1.0)

distance_mat  = 1 - np.abs(spearman_vals)
np.fill_diagonal(distance_mat, 0.0)
distance_mat  = np.clip(distance_mat, 0, 1)

# Ensure symmetry
distance_mat  = (distance_mat + distance_mat.T) / 2

condensed     = squareform(distance_mat, checks=False)
linkage_mat   = linkage(condensed, method="ward")

# ─── Optimal cluster count via Silhouette score ───────────────────────────────
from sklearn.metrics import silhouette_score

print("  Evaluating silhouette scores for k = 5 to 20...")
sil_scores = {}

for k in range(5, 21):
    labels = fcluster(linkage_mat, k, criterion="maxclust")
    try:
        score  = silhouette_score(distance_mat, labels, metric="precomputed")
        sil_scores[k] = round(score, 4)
    except Exception:
        sil_scores[k] = np.nan

best_k   = max(sil_scores, key=lambda k: sil_scores[k] if not np.isnan(sil_scores[k]) else -1)
best_sil = sil_scores[best_k]

print(f"\n  Silhouette scores:")
for k, s in sil_scores.items():
    marker = "  ◀ best" if k == best_k else ""
    print(f"    k={k:2d}  →  {s:.4f}{marker}")

print(f"\n  Optimal k : {best_k}  (silhouette = {best_sil:.4f})")

# ─── Assign cluster labels ────────────────────────────────────────────────────
cluster_labels = fcluster(linkage_mat, best_k, criterion="maxclust")
cluster_df     = pd.DataFrame({
    "coin_id"   : coins,
    "cluster_id": cluster_labels,
})

# ─── Cluster size distribution ────────────────────────────────────────────────
cluster_sizes = cluster_df["cluster_id"].value_counts().sort_index()
print(f"\n  Cluster size distribution (k={best_k}):")
for cid, size in cluster_sizes.items():
    members = cluster_df.loc[cluster_df["cluster_id"] == cid, "coin_id"].tolist()
    print(f"    Cluster {cid:2d} : {size:3d} coins  "
          f"│ {', '.join(members[:5])}{'...' if size > 5 else ''}")

# ─── Dendrogram ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(24, 8))
dendrogram(
    linkage_mat,
    labels=coins,
    leaf_rotation=90,
    leaf_font_size=6,
    color_threshold=linkage_mat[-(best_k - 1), 2],
    ax=ax
)
ax.set_title(f"Hierarchical Clustering Dendrogram — Spearman Distance (k={best_k})",
             fontsize=13)
ax.set_ylabel("Ward Distance")
plt.tight_layout()
plt.savefig("outputs/figures/phase2_dendrogram.png", dpi=150)
plt.close()
print("\n  ✅ Saved: phase2_dendrogram.png")


# ═══════════════════════════════════════════════════════════════════════════════
# SAVE OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Saving Phase 2 outputs ──────────────────────────────────────────────────")

# 1. Full correlation matrices (Pearson + Spearman stacked)
corr_combined = {
    "pearson" : pearson_full,
    "spearman": spearman_full,
}
for name, mat in corr_combined.items():
    mat.to_parquet(f"outputs/04_correlation_{name}_full.parquet")
print("  ✅ 04_correlation_pearson_full.parquet  + spearman")

# 2. BTC-adjusted correlation
spearman_adj.to_parquet("outputs/05_correlation_matrix_btcadj.parquet")
print("  ✅ 05_correlation_matrix_btcadj.parquet")

# 3. Rolling endpoint matrices
rolling_30d.to_parquet("outputs/phase2_rolling_corr_30d.parquet")
rolling_90d.to_parquet("outputs/phase2_rolling_corr_90d.parquet")
print("  ✅ phase2_rolling_corr_30d.parquet  + 90d")

# 4. Subperiod correlation matrices
for period, mat in subperiod_corr.items():
    mat.to_parquet(f"outputs/phase2_subperiod_corr_{period}.parquet")
print("  ✅ phase2_subperiod_corr_Period_A/B/C/D.parquet")

# 5. Stability analysis
stability_df.to_csv("outputs/phase2_correlation_stability.csv", index=False)
print("  ✅ phase2_correlation_stability.csv")

# 6. First-pass cluster assignments
cluster_df.to_csv("outputs/phase2_clustering_baseline.csv", index=False)
print("  ✅ phase2_clustering_baseline.csv")

# 7. BTC betas
beta_df.to_csv("outputs/phase2_btc_betas.csv", index=False)
print("  ✅ phase2_btc_betas.csv")

# ─── Final summary ────────────────────────────────────────────────────────────
print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  PHASE 2 COMPLETE                                                ║
╠══════════════════════════════════════════════════════════════════╣
║  BTC–ETH Spearman ρ         : {btc_eth_spearman:.4f}                         ║
║  Median pairwise ρ (full)   : {np.median(upper_tri[~np.isnan(upper_tri)]):.4f}                         ║
║  BH-sig pairs (Spearman)    : {n_sig_s:,} / {len(upper_tri):,}               ║
║  Median ρ drop (BTC-adj)    : {np.median(upper_raw[mask_both] - upper_adj[mask_both]):.4f}                         ║
║  Baseline clusters (k)      : {best_k}  (silhouette = {best_sil:.4f})          ║
║  Subperiods analysed        : 4  (A / B / C / D)                ║
╠══════════════════════════════════════════════════════════════════╣
║  Next: Phase 3 — Copula-Based Tail Dependence                   ║
║  Key inputs for Phase 3:                                         ║
║    02_returns_matrix.parquet                                     ║
║    phase1_fat_tail_params.csv   ← ν values for copula selection  ║
║    phase2_clustering_baseline   ← prioritise high-corr pairs     ║
╚══════════════════════════════════════════════════════════════════╝
""")

Loading Phase 1 outputs...
  Coins            : 125
  Return matrix    : (729, 125)
  Date range       : 2024-03-22 00:00:00  →  2026-03-20 00:00:00

── Consolidating subperiods ────────────────────────────────────────────────
  Period_A : 224 days  (2024-03-22 → 2024-10-31)
  Period_B :  92 days  (2024-11-01 → 2025-01-31)
  Period_C : 242 days  (2025-02-01 → 2025-09-30)
  Period_D : 171 days  (2025-10-01 → 2026-03-20)

── Phase 2.1 │ Static Correlation Matrices ─────────────────────────────────
  Computing full-period Pearson + Spearman...

  Sanity check — BTC–ETH correlation:
    Pearson  : 0.8210  (expected > 0.70)
    Spearman : 0.7835  (expected > 0.70)
  ✅ Sanity check passed

  Spearman correlation distribution (all 7,750 pairs):
    Mean   : 0.6545
    Median : 0.6832
    Std    : 0.1428
    Min    : -0.0402
    Max    : 0.9412
    Pairs ρ > 0.70  : 3,274
    Pairs ρ > 0.50  : 6,975
    Pairs ρ < 0.20  : 246

  BH-significant pairs — Pearson  : 7,650 / 7,750
  BH-significant p